# 初期化

起動時はまずこれを実行する

In [ ]:
# ライブラリのインストールと読み込み
!pip install bs4
from urllib import parse
import requests
from bs4 import BeautifulSoup
import json

# Bookmeter

In [ ]:
def get_soup(url: str):
  """url先のbs4のオブジェクトを返す"""
  response = requests.get(url)
  soup = BeautifulSoup(response.text)
  return soup

def get_bookmeter_review(keyword: str):
  """ブックメーターのレビューを最大40件取得"""
  url = f"https://bookmeter.com/search?author=&keyword={parse.quote(keyword)}&partial=true&sort=recommended&type=japanese_v2&page=1"

  soup = get_soup(url)
  target_book = soup.findAll("div", class_="detail__title")[0].find("a", href=True)
  id = target_book["href"].split("/")[2]
  title = target_book.text

  header= {"content-type": "application/json"}
  review = requests.get(f"https://bookmeter.com/books/{id}/reviews.json?offset=0&limit=40", headers=header)
  review_json = review.json()

  return {"title": title, "reviews": [item["content"] for item in review_json["resources"]]}


reviews = get_bookmeter_review("ある閉ざされた雪の山荘で")
# 返り値は連想配列
# title:   取得した本のタイトル
# reviews: レビューの配列

print(f"タイトル: {reviews['title']}\n")
for r in reviews["reviews"]:
  print(r)

# Amazon

In [ ]:
def get_soup(url: str):
  """url先のbs4のオブジェクトを返す"""
  # アマゾンはスクレイピング対策されているのでブラウザのふりをするためにヘッダーを設定
  headers = {
    "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "accept-language": "ja-JP,ja;q=0.9,en-US;q=0.8,en;q=0.7",
    "cache-control": "no-cache",
    "device-memory": "8",
    "downlink": "10",
    "dpr": "3",
    "ect": "4g",
    "pragma": "no-cache",
    "priority": "u=0, i",
    "rtt": "50",
    "sec-ch-device-memory": "8",
    "sec-ch-dpr": "3",
    "sec-ch-ua": "\"Google Chrome\";v=\"131\", \"Chromium\";v=\"131\", \"Not_A Brand\";v=\"24\"",
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": "\"Windows\"",
    "sec-ch-ua-platform-version": "\"19.0.0\"",
    "sec-ch-viewport-width": "725",
    "sec-fetch-dest": "document",
    "sec-fetch-mode": "navigate",
    "sec-fetch-site": "same-origin",
    "sec-fetch-user": "?1",
    "upgrade-insecure-requests": "1",
    "viewport-width": "725"
  }
  response = requests.get(url, headers=headers)
  soup = BeautifulSoup(response.text)
  return soup

def get_amazon_review(keyword: str):
  url = f"https://www.amazon.co.jp/s?k={parse.quote(keyword)}&rh=n%3A465392"

  soup = get_soup(url)
  path = soup.find_all("div", class_="template=SEARCH_RESULTS")[0].find("a", href=True)["href"]

  book_url = f"https://www.amazon.co.jp{path}"
  reviews_soup = get_soup(book_url)
  reviews = [item.find("span").text for item in reviews_soup.find_all("div", class_="reviewText")]
  title = reviews_soup.find(id = "productTitle").text.strip()
  return {"title": title, "reviews": reviews}


reviews = get_amazon_review("ある閉ざされた雪の山荘で")

print(f"タイトル: {reviews['title']}")
for r in reviews["reviews"]:
  print(r)